In [1]:
import os
import math
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

def lonlat_to_utm_batch(in_folder, out_folder=None, lon_col="lon", lat_col="lat"):
    """
    将文件夹中所有 CSV 的 WGS84 坐标转换为 UTM。
    每个文件生成新的 *_utm.csv 文件，不覆盖原始文件。
    """

    if out_folder is None:
        out_folder = os.path.join(in_folder, "2022_12_utm_out")
        """
        改变这里 来设置输出文件夹的名字
        """

    os.makedirs(out_folder, exist_ok=True)

    csv_files = [f for f in os.listdir(in_folder) if f.lower().endswith(".csv")]
    if not csv_files:
        print("❌ 文件夹中没有找到 CSV 文件。")
        return

    for fname in csv_files:
        in_path = os.path.join(in_folder, fname)
        out_path = os.path.join(out_folder, "UTM_" + fname)

        # 读取 CSV
        df = pd.read_csv(in_path)

        if lon_col not in df.columns or lat_col not in df.columns:
            print(f"⚠️ 跳过 {fname}：缺少 '{lon_col}' 或 '{lat_col}' 列。")
            continue

        # 根据中位数经纬度判断UTM带
        lon0 = float(df[lon_col].median())
        lat0 = float(df[lat_col].median())
        zone = int(math.floor((lon0 + 180) / 6) + 1)
        epsg = 32600 + zone if lat0 >= 0 else 32700 + zone

        # 转为GeoDataFrame并投影
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
            crs="EPSG:4326"
        ).to_crs(f"EPSG:{epsg}")

        # 添加新列
        df["easting"] = gdf.geometry.x
        df["northing"] = gdf.geometry.y
        df["utm_zone"] = zone
        df["epsg"] = epsg

        # 保存新文件
        df.to_csv(out_path, index=False)
        print(f"✅ 已生成: {out_path}  (EPSG:{epsg}, Zone {zone})")




In [2]:
# 输入：存放原始CSV的文件夹
in_folder = "/Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_12_ICESat2_csv"
lonlat_to_utm_batch(in_folder)

✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_12_ICESat2_csv/2022_12_utm_out/UTM_ATL08_20221212070923_12611702_007_01_Rule6_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_12_ICESat2_csv/2022_12_utm_out/UTM_ATL08_20221212070923_12611702_007_01_Rule7_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_12_ICESat2_csv/2022_12_utm_out/UTM_ATL08_20221212070923_12611702_007_01_Rule1_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_12_ICESat2_csv/2022_12_utm_out/UTM_ATL08_20221212070923_12611702_007_01_Rule5_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_12_ICESat2_csv/2022_12_utm_out/UTM_ATL08_20221212070923_12611702_007_01_Rule4_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/2022_